# Golden Records Validation
Confirms that every record in `bbh_golden_tasks_clean.json` is truly Golden:
- **Few-shot (Teacher)** prediction = Correct Answer ✅
- **Zero-shot (Student)** prediction ≠ Correct Answer ❌

Produces:
1. A summary comparison table for all 113 records
2. A full per-record audit with Teacher Prompt, Student Prompt, and Correct Answer

In [1]:
!pip install transformers torch -q

In [2]:
import json
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"
).to(device)
model.eval()
print("Model loaded.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Model loaded.


In [3]:
A_ID = tokenizer.encode("A", add_special_tokens=False)[0]
B_ID = tokenizer.encode("B", add_special_tokens=False)[0]
C_ID = tokenizer.encode("C", add_special_tokens=False)[0]
D_ID = tokenizer.encode("D", add_special_tokens=False)[0]
MC_IDS = [A_ID, B_ID, C_ID, D_ID]
MC_LETTERS = ["A", "B", "C", "D"]

def format_prompt(text: str) -> str:
    messages = [{"role": "user", "content": text}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) + "Answer:"

def predict_mc(prompt_text: str):
    formatted = format_prompt(prompt_text)
    inputs = tokenizer(formatted, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :].float()
        probs = F.softmax(logits, dim=-1)
    mc_probs = [probs[0, tid].item() for tid in MC_IDS]
    best_idx = mc_probs.index(max(mc_probs))
    return MC_LETTERS[best_idx]

In [6]:
with open("bbh_golden_tasks_clean.json", "r") as f:
    records = json.load(f)

print(f"Loaded {len(records)} clean golden records.")
print("Running validation forward passes...\n")

results = []
for rec in tqdm(records, desc="Validating"):
    teacher_pred = predict_mc(rec["teacher_prompt"])
    student_pred = predict_mc(rec["student_prompt"])
    results.append({
        "id": rec["id"],
        "task": rec["task"],
        "correct_answer": rec["answer"],
        "fewshot_pred": teacher_pred,
        "zeroshot_pred": student_pred,
        "teacher_prompt": rec["teacher_prompt"],
        "student_prompt": rec["student_prompt"],
        "is_golden": teacher_pred == rec["answer"] and student_pred != rec["answer"]
    })

print("Done.")

Loaded 113 clean golden records.
Running validation forward passes...



Validating: 100%|██████████| 113/113 [01:04<00:00,  1.76it/s]

Done.


In [7]:
# ── Summary Comparison Table ──
truly_golden = sum(1 for r in results if r["is_golden"])
corrupted    = len(results) - truly_golden

print(f"{'='*80}")
print(f"VALIDATION SUMMARY")
print(f"{'='*80}")
print(f"Total Records   : {len(results)}")
print(f"Truly Golden    : {truly_golden}  (teacher ✅, student ❌)")
print(f"Corrupted/Stale : {corrupted}  (failed golden condition — inspect below)")
print(f"{'='*80}\n")

# Column widths
header = f"{'ID':>6} | {'Task':<40} | {'Correct':>7} | {'Few-shot':>8} | {'Zero-shot':>9} | {'Golden?':>7}"
print(header)
print("-" * len(header))

for r in results:
    teacher_mark = "✅" if r["fewshot_pred"] == r["correct_answer"] else "❌"
    student_mark = "✅" if r["zeroshot_pred"] == r["correct_answer"] else "❌"
    golden_mark  = "✅" if r["is_golden"] else "⚠️"
    print(
        f"{r['id']:>6} | {r['task']:<40} | {r['correct_answer']:>7} | "
        f"{r['fewshot_pred']:>6}{teacher_mark} | {r['zeroshot_pred']:>7}{student_mark} | {golden_mark:>7}"
    )

VALIDATION SUMMARY
Total Records   : 113
Truly Golden    : 113  (teacher ✅, student ❌)
Corrupted/Stale : 0  (failed golden condition — inspect below)

    ID | Task                                     | Correct | Few-shot | Zero-shot | Golden?
--------------------------------------------------------------------------------------------
    55 | boolean_expressions                      |       A |      A✅ |       B❌ |       ✅
  1117 | sports_understanding                     |       A |      A✅ |       B❌ |       ✅
  1212 | sports_understanding                     |       A |      A✅ |       B❌ |       ✅
  1229 | sports_understanding                     |       A |      A✅ |       B❌ |       ✅
  1138 | sports_understanding                     |       A |      A✅ |       B❌ |       ✅
  1150 | sports_understanding                     |       A |      A✅ |       B❌ |       ✅
  1166 | sports_understanding                     |       A |      A✅ |       B❌ |       ✅
  1102 | sports_understand

In [8]:
# ── Full Per-Record Audit ──
DIVIDER = "=" * 80

for i, r in enumerate(results):
    golden_label = "[GOLDEN ✅]" if r["is_golden"] else "[⚠️ NOT GOLDEN]"
    print(f"\n{DIVIDER}")
    print(f"Record {i+1:>3} / {len(results)} | ID: {r['id']} | Task: {r['task']} | {golden_label}")
    print(f"Correct Answer : {r['correct_answer']}  |  Few-shot Pred: {r['fewshot_pred']}  |  Zero-shot Pred: {r['zeroshot_pred']}")
    print(DIVIDER)
    print("\n--- TEACHER PROMPT (2-shot) ---")
    print(r["teacher_prompt"])
    print("\n--- STUDENT PROMPT (0-shot) ---")
    print(r["student_prompt"])
    print(DIVIDER)


Record   1 / 113 | ID: 55 | Task: boolean_expressions | [GOLDEN ✅]
Correct Answer : A  |  Few-shot Pred: A  |  Zero-shot Pred: B

--- TEACHER PROMPT (2-shot) ---
Task: Evaluate the Boolean expression.

Example 1
not ( True ) and ( True ) is
Options:
A) True
B) False
Answer: B

Example 2
True and not not ( not False ) is
Options:
A) False
B) True
Answer: B

Question
( not False and ( True ) ) is
Options:
A) True
B) False

--- STUDENT PROMPT (0-shot) ---
Task: Evaluate the Boolean expression.

Question
( not False and ( True ) ) is
Options:
A) True
B) False

Record   2 / 113 | ID: 1117 | Task: sports_understanding | [GOLDEN ✅]
Correct Answer : A  |  Few-shot Pred: A  |  Zero-shot Pred: B

--- TEACHER PROMPT (2-shot) ---
Task: Determine whether the sports statement is plausible or implausible.

Example 1
Is the following sentence plausible? "Elias Lindholm beat the buzzer."
Options:
A) yes
B) no
Answer: B

Example 2
Is the following sentence plausible? "John Carlson scored in the third p